In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

In [18]:
np.random.seed(42)
tf.random.set_seed(42)

In [19]:
train = pd.read_parquet('../data/cleanedTrain.parquet')

In [20]:
train.sort_values('timestamp', inplace=True)

In [21]:
train.replace([np.inf, -np.inf], np.nan, inplace=True)
train.fillna(train.median(), inplace=True)

In [22]:
feature_cols = [col for col in train.columns if col not in ['timestamp', 'label']]
print(train[feature_cols].dtypes.value_counts())

scaler = StandardScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols]).astype(np.float32)
train['label'] = train['label'].astype(np.float32)
print(train[feature_cols].dtypes.value_counts())

float64    895
Name: count, dtype: int64


d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\Anaconda\Lib\site-packages\sklearn\utils\extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


float32    895
Name: count, dtype: int64


In [23]:
print(train[feature_cols].isnull().sum().sum())


11043627


In [24]:
train[feature_cols] = train[feature_cols].fillna(0)


In [25]:
X = train[feature_cols]
y = train['label']

selector = SelectKBest(score_func=f_regression, k=250)
X_selected = selector.fit_transform(X, y)

selected_features = [feature_cols[i] for i in selector.get_support(indices=True)]
feature_cols = selected_features
print(f"Selected {len(feature_cols)} features")

Selected 250 features


In [26]:
train = train.copy()


In [27]:
train['volume_bin'], bin_edges = pd.qcut(train['volume'], q=3, labels=['low', 'mid', 'high'], retbins=True)

In [28]:
def create_sequences(df, timesteps=10):
    Xs, ys = [], []
    for i in range(len(df) - timesteps):
        seq = df[feature_cols].iloc[i:i+timesteps].values.astype(np.float32)
        print(seq.shape)  
        Xs.append(seq)
        ys.append(df['label'].iloc[i+timesteps])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


In [29]:
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(128, input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model


In [30]:
class CryptoSequence(Sequence):
    def __init__(self, df, features, timesteps=5, batch_size=256):
        self.df = df.reset_index(drop=True)
        self.features = features
        self.timesteps = timesteps
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.df) - timesteps)

    def __len__(self):
        return int(np.floor(len(self.indexes) / self.batch_size))

    def __getitem__(self, idx):
        batch_start = idx * self.batch_size
        batch_end = batch_start + self.batch_size
        batch_indexes = self.indexes[batch_start:batch_end]

        X_batch = np.array([
            self.df.loc[i:i+self.timesteps-1, self.features].values 
            for i in batch_indexes
        ], dtype=np.float32)

        y_batch = self.df.loc[batch_indexes + self.timesteps, 'label'].values.astype(np.float32)

        return X_batch, y_batch

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)

In [32]:
timesteps = 5
scores = {}

for bin_label in ['low', 'mid', 'high']:
    print(f"\n📊 Training for volume bin: {bin_label.upper()}")
    
    df_bin = train[train['volume_bin'] == bin_label].reset_index(drop=True)
    
    if len(df_bin) < timesteps + 100:
        print("❗ Skipped — not enough data")
        continue
    
    split_idx = int(0.8 * (len(df_bin) - timesteps))
    train_df, val_df = df_bin.iloc[:split_idx+timesteps], df_bin.iloc[split_idx:]
    
    train_gen = CryptoSequence(train_df, feature_cols, timesteps=timesteps, batch_size=256)
    val_gen = CryptoSequence(val_df, feature_cols, timesteps=timesteps, batch_size=256)
    
    model = build_lstm((timesteps, len(feature_cols)))

    
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=20,        
        callbacks=[early_stop], 
        verbose=2
    )
    
    y_val, y_pred = [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch).flatten()
        y_val.extend(y_batch)
        y_pred.extend(preds)
    
    corr, _ = pearsonr(y_val, y_pred)
    print(f"Validation Pearson: {corr:.5f}")
    
    model.save(f'../results/LSTM/LSTM_{bin_label}_volume.keras')
    scores[bin_label] = corr



📊 Training for volume bin: LOW


d:\Anaconda\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
d:\Anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


MemoryError: Unable to allocate 134. MiB for an array with shape (250, 140239) and data type float32